# Cylinder Monitor — Signal Analysis Workbench v2

Load or **record** a WAV, visualize both channels, tune detection parameters, log sessions, and track degradation trends.

**Workflow — single session:**
1. Run **Imports** cell
2. Run **Config** cell — set parameters
3. Either **Record** a new file OR set `WAV_FILE` and run **Load**
4. Run all analysis cells top to bottom
5. Run **Save WAV** if you recorded live
6. Run **Log Session** to append results to CSV

**Workflow — trend analysis (multi-session):**
1. After several sessions are logged, run **Trend Plot**, **CUSUM**, and **Inter-Cylinder Ratio** cells

**New in v2:** Record from mic · Save WAV · Audio playback · Coincidence quality score · Session CSV logging · Trend plot · CUSUM alert · Inter-cylinder ratio

## Imports

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import scipy.signal as signal
import scipy.stats as stats
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display, HTML, Audio
import pandas as pd
import datetime
import csv

try:
    from pydub import AudioSegment
    PYDUB_OK = True
except ImportError:
    PYDUB_OK = False
    print('pydub not installed — M4A/MP3 loading unavailable. WAV and FLAC still work.')

try:
    import sounddevice as sd
    import soundfile as sf
    AUDIO_IO_OK = True
except ImportError:
    AUDIO_IO_OK = False
    print('sounddevice / soundfile not installed — record and save unavailable.')
    print('Install: pip install sounddevice soundfile')

pio.renderers.default = 'iframe'

DARK = dict(
    paper_bgcolor='#1a1a1a', plot_bgcolor='#111111',
    font_color='#cccccc',
    xaxis=dict(gridcolor='#2a2a2a', zerolinecolor='#444'),
    yaxis=dict(gridcolor='#2a2a2a', zerolinecolor='#444'),
)

# Notebook lives in analysis/ — repo root is one level up
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
(REPO_ROOT / 'test_data').mkdir(exist_ok=True)
(REPO_ROOT / 'session_logs').mkdir(exist_ok=True)

print('Packages loaded OK')
print(f'Repo root: {REPO_ROOT}')

## Config — set your filename and parameters here
These values mirror the app Settings tab exactly. Tune here, then copy suggested values back to the app.

In [ ]:
# ── File ──────────────────────────────────────────────────────────────────────
WAV_FILE = str(REPO_ROOT / 'test_data' / 'Test123.m4a')  # <-- change filename

# ── Session metadata ──────────────────────────────────────────────────────────
CYLINDER_ID = 1          # which cylinder (1–6)
PSI         = 80.0       # supply pressure at time of measurement — required
NOTES       = ''         # optional free-text notes for this session

# ── Detection parameters (mirror the app Settings tab) ───────────────────────
STROKE_IN        = 1.0    # stroke length in inches
IMPACT_MULT      = 10     # threshold multiplier for T_end detection
BREAKAWAY_MULT   = 3      # threshold multiplier for T_start search
DEBOUNCE_MS      = 50     # minimum ms between two spikes
MIN_LOOKBACK_MS  = 15     # minimum ms before T_end to search for T_start
MAX_LOOKBACK_MS  = 100    # maximum ms before T_end to search for T_start
HF_BIN_LOW       = 10     # FFT bin low  (~1 kHz at 48kHz/480 samples)
HF_BIN_HIGH      = 100    # FFT bin high (~10 kHz)
HF_FLOOR         = 0.01   # minimum HF energy to pass the FFT gate
BASELINE_PCT     = 50     # percentile of RMS history used as baseline
CHUNK_MS         = 10     # RMS chunk size in ms (must match app)

# ── Signal combination method ─────────────────────────────────────────────────
# 'additive'       |Ch0| + |Ch1|  — use for laptop/phone mic
# 'multiplicative' |Ch0| x |Ch1|  — use for wireless mics mounted on cylinder
METHOD = 'additive'

# ── Recording settings (Record cell only) ────────────────────────────────────
RECORD_SECONDS   = 5      # how long to record
RECORD_SR        = 48000  # sample rate — must match CM28 output
RECORD_CHANNELS  = 2      # 2 = stereo (both CM28 transmitters)
DEVICE_INDEX     = None   # None = system default. Run List Devices cell to find CM28 index

# ── Session log CSV ───────────────────────────────────────────────────────────
SESSION_LOG_CSV  = str(REPO_ROOT / 'session_logs' / 'cylinder_sessions.csv')

# ── CUSUM parameters ─────────────────────────────────────────────────────────
CUSUM_K = 0.5   # slack parameter — sensitivity vs false alarm tradeoff
CUSUM_H = 5.0   # decision threshold — raise to reduce false alarms

print('Config set.')
print(f'  Cylinder: {CYLINDER_ID}  |  PSI: {PSI}  |  Method: {METHOD}')
print(f'  File: {WAV_FILE}')
print(f'  Log:  {SESSION_LOG_CSV}')

## Record from Mic — live capture
Lists available audio devices, then records from the selected device.
Skip this cell if you are loading a pre-recorded file.

In [ ]:
# ── Step 1: List available audio devices ─────────────────────────────────────
# Run this cell first to find your BY-V30 device index.
# Look for the USB audio device — it will say something like 'USB Audio Device'
# or 'Boya'. Set DEVICE_INDEX in Config to that number.

if not AUDIO_IO_OK:
    print('sounddevice not installed. Run: pip install sounddevice soundfile')
else:
    devices = sd.query_devices()
    print(f'{"Index":>5}  {"Name":<45}  {"In ch":>5}  {"Out ch":>6}  {"Default SR":>10}')
    print('-' * 80)
    for i, d in enumerate(devices):
        if d['max_input_channels'] > 0:
            marker = ' <-- DEFAULT' if i == sd.default.device[0] else ''
            print(f"{i:>5}  {d['name']:<45}  {d['max_input_channels']:>5}  "
                  f"{d['max_output_channels']:>6}  {int(d['default_samplerate']):>10}{marker}")
    print()
    print('Set DEVICE_INDEX in Config to the index of your BY-V30 / USB mic.')
    print('Blue light on BY-V30 transmitters = NR OFF (required).')

In [ ]:
if not AUDIO_IO_OK:
    print('sounddevice not installed — skipping record.')
else:
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    auto_filename = str(REPO_ROOT / 'test_data' / f'cyl{CYLINDER_ID}_{ts}.wav')

    print(f'Recording {RECORD_SECONDS}s at {RECORD_SR} Hz, {RECORD_CHANNELS} ch ...')
    print(f'Device index: {DEVICE_INDEX} (None = system default)')
    print('FIRE NOW')

    recorded_data = sd.rec(
        int(RECORD_SECONDS * RECORD_SR),
        samplerate=RECORD_SR,
        channels=RECORD_CHANNELS,
        dtype='float32',
        device=DEVICE_INDEX
    )
    sd.wait()
    print(f'Done. Shape: {recorded_data.shape}  Peak: {np.max(np.abs(recorded_data)):.4f}')

    data = recorded_data
    ch0  = data[:, 0]
    ch1  = data[:, 1] if data.shape[1] > 1 else data[:, 0]
    sr   = RECORD_SR
    duration_s = len(ch0) / sr
    t    = np.linspace(0, duration_s, len(ch0))

    RECORDED_FILEPATH = auto_filename
    print(f'Will save to: {auto_filename}  (run Save WAV cell to write)')

## Load Audio File — WAV, M4A, FLAC
Skip this cell if you just recorded using the Record cell above.

In [ ]:
path   = Path(WAV_FILE)
suffix = path.suffix.lower()

if not path.exists():
    raise FileNotFoundError(
        f'File not found: {path.resolve()}\n'
        f'Drop a file into test_data/ and update WAV_FILE in Config.'
    )

if suffix in ('.m4a', '.mp4', '.aac', '.ogg', '.mp3'):
    if not PYDUB_OK:
        raise ImportError('pydub required for M4A/MP3. Run: pip install pydub')
    audio   = AudioSegment.from_file(path)
    sr      = audio.frame_rate
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
    if audio.channels == 2:
        samples = samples.reshape(-1, 2)
    else:
        samples = np.stack([samples, samples], axis=1)
        print('Mono file — duplicated to stereo')
    max_val = float(2 ** (8 * audio.sample_width - 1))
    data    = samples / max_val
elif suffix == '.flac' and AUDIO_IO_OK:
    data, sr = sf.read(path, dtype='float32', always_2d=True)
else:
    sr, data = wav.read(path)
    if data.dtype == np.int16:
        data = data.astype(np.float32) / 32768.0
    elif data.dtype == np.int32:
        data = data.astype(np.float32) / 2147483648.0
    else:
        data = data.astype(np.float32)
    if data.ndim == 1:
        data = np.stack([data, data], axis=1)
        print('Mono file — duplicated to stereo')

ch0 = data[:, 0]
ch1 = data[:, 1]
duration_s = len(ch0) / sr
t = np.linspace(0, duration_s, len(ch0))

print(f'File:      {path.name}')
print(f'Format:    {suffix}')
print(f'Rate:      {sr} Hz')
print(f'Duration:  {duration_s*1000:.1f} ms  ({duration_s:.2f} s)')
print(f'Samples:   {len(ch0):,}')
print(f'Ch0 peak:  {np.max(np.abs(ch0)):.4f}')
print(f'Ch1 peak:  {np.max(np.abs(ch1)):.4f}')
if np.max(np.abs(ch1)) < 0.001:
    print('WARNING: Ch1 is nearly silent — possible mono mix from BY-V30. Run acceptance test.')

## Playback — listen in the notebook
Your ear catches dropouts, buzzes, and double-clicks that plots miss.
Ch0 and Ch1 played separately so you can confirm both sensors received signal.

In [ ]:
print('Ch0 (Left / Sensor A):')
display(Audio(ch0, rate=sr))
print('Ch1 (Right / Sensor B):')
display(Audio(ch1, rate=sr))

## Save WAV — write recording to disk
Only needed after a live recording session. Skip if you loaded a file.

In [ ]:
if not AUDIO_IO_OK:
    print('soundfile not installed — skipping save. Run: pip install soundfile')
else:
    # Uses auto-generated filename from Record cell, or override here:
    save_path = RECORDED_FILEPATH if 'RECORDED_FILEPATH' in dir() else \
                f'test_data/cyl{CYLINDER_ID}_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.wav'

    sf.write(save_path, data, sr, subtype='PCM_16')
    print(f'Saved: {save_path}')
    print(f'Size:  {Path(save_path).stat().st_size / 1024:.1f} KB')

## Raw Waveforms — both channels
First visual sanity check. If Ch1 is flat, the sensor was not coupled or NR was ON.

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Ch0 — Sensor A (Left)', 'Ch1 — Sensor B (Right)'))
fig.add_trace(go.Scatter(x=t*1000, y=ch0, mode='lines',
              line=dict(color='#4fc3f7', width=0.8), name='Ch0'), row=1, col=1)
fig.add_trace(go.Scatter(x=t*1000, y=ch1, mode='lines',
              line=dict(color='#81d4fa', width=0.8), name='Ch1'), row=2, col=1)
fig.update_layout(title='Raw Waveforms', height=500, **DARK)
fig.update_xaxes(title_text='Time (ms)', row=2, col=1, gridcolor='#2a2a2a')
fig.update_yaxes(gridcolor='#2a2a2a')
fig.show()

## Combined Signal — additive vs multiplicative
Additive is robust at all amplitudes. Multiplicative suppresses noise that hits only one sensor.
Compare both — if they agree on spike location, confidence is high.

In [ ]:
combined_add  = np.abs(ch0) + np.abs(ch1)
combined_mult = np.abs(ch0) * np.abs(ch1)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Additive |Ch0|+|Ch1|', 'Multiplicative |Ch0|×|Ch1|'))
fig.add_trace(go.Scatter(x=t*1000, y=combined_add,  mode='lines',
              line=dict(color='#66bb6a', width=0.8), name='Additive'), row=1, col=1)
fig.add_trace(go.Scatter(x=t*1000, y=combined_mult, mode='lines',
              line=dict(color='#ffa726', width=0.8), name='Multiplicative'), row=2, col=1)
fig.update_layout(title='Combined Signal — Additive vs Multiplicative', height=500, **DARK)
fig.update_xaxes(title_text='Time (ms)', row=2, col=1, gridcolor='#2a2a2a')
fig.update_yaxes(gridcolor='#2a2a2a')
fig.show()

## Rolling RMS — what the app sees
Exact replica of the AudioWorklet computation. The threshold lines show what the app
will trigger on at the current multiplier settings.

In [ ]:
combined      = combined_add if METHOD == 'additive' else combined_mult
chunk_samples = int(CHUNK_MS / 1000 * sr)

n_chunks = len(combined) // chunk_samples
rms_vals = np.array([
    np.sqrt(np.mean(combined[i*chunk_samples:(i+1)*chunk_samples]**2))
    for i in range(n_chunks)
])
t_chunks = np.array([(i + 0.5) * CHUNK_MS for i in range(n_chunks)])

baseline  = float(np.percentile(rms_vals, BASELINE_PCT))
threshold = baseline * IMPACT_MULT
bkwy_thr  = baseline * BREAKAWAY_MULT

print(f'Method:    {METHOD}')
print(f'Chunks:    {n_chunks}  ({CHUNK_MS}ms each)')
print(f'Baseline:  {baseline:.6f}  ({BASELINE_PCT}th percentile)')
print(f'Threshold: {threshold:.6f}  ({IMPACT_MULT}× baseline)')
print(f'Breakaway: {bkwy_thr:.6f}  ({BREAKAWAY_MULT}× baseline)')

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_chunks, y=rms_vals, mode='lines',
              line=dict(color='#4fc3f7', width=1), name='RMS'))
fig.add_hline(y=threshold, line=dict(color='#ef5350', width=1.5, dash='dash'),
              annotation_text=f'Impact threshold ({IMPACT_MULT}×)')
fig.add_hline(y=bkwy_thr,  line=dict(color='#ffa726', width=1, dash='dot'),
              annotation_text=f'Breakaway ({BREAKAWAY_MULT}×)')
fig.add_hline(y=baseline,  line=dict(color='#555555', width=1),
              annotation_text=f'Baseline ({BASELINE_PCT}th pct)')
fig.update_layout(title='Rolling RMS — what the app sees',
                  xaxis_title='Time (ms)', yaxis_title='RMS', height=400, **DARK)
fig.show()

## Coincidence Quality Score — per-event inter-channel delay
Structure-borne vibration from the cylinder hits both sensors simultaneously (< 0.1ms delay).
External factory noise hits one sensor first with a measurable delay.
This cell computes a quality score per detected spike — high score = real cylinder event.
Produces a continuous quality metric rather than binary pass/fail.

In [ ]:
def inter_channel_delay_ms(ch0, ch1, center_sample, window_samples, sr):
    """Cross-correlate a window around a spike to find inter-channel delay."""
    half = window_samples // 2
    s = max(0, center_sample - half)
    e = min(len(ch0), center_sample + half)
    seg0 = ch0[s:e]
    seg1 = ch1[s:e]
    if len(seg0) < 4 or np.max(np.abs(seg0)) < 1e-8 or np.max(np.abs(seg1)) < 1e-8:
        return 0.0, 0.0
    corr = np.correlate(seg0 - seg0.mean(), seg1 - seg1.mean(), mode='full')
    lag_samples = np.argmax(np.abs(corr)) - (len(seg0) - 1)
    delay_ms    = lag_samples / sr * 1000
    # Quality: 1.0 = perfectly simultaneous, 0.0 = max possible delay
    max_delay_ms = (window_samples / 2) / sr * 1000
    quality = max(0.0, 1.0 - abs(delay_ms) / max_delay_ms)
    return delay_ms, quality

WINDOW_MS       = 10    # window around each spike for cross-correlation
QUALITY_WARN    = 0.5   # quality below this = flag as likely noise
window_samples  = int(WINDOW_MS / 1000 * sr)

# We need spikes — run after Spike Detection cell or compute a quick pass here
# This uses the rms_vals and threshold already computed above
quick_spikes = []
last_i = -999
for i in range(n_chunks):
    if rms_vals[i] >= threshold and (i - last_i) >= DEBOUNCE_MS / CHUNK_MS:
        center = int((i + 0.5) * chunk_samples)
        delay_ms, quality = inter_channel_delay_ms(ch0, ch1, center, window_samples, sr)
        quick_spikes.append({
            'i': i, 't_ms': i * CHUNK_MS,
            'rms': rms_vals[i], 'delay_ms': delay_ms, 'quality': quality
        })
        last_i = i

if not quick_spikes:
    print('No spikes found at current threshold — lower IMPACT_MULT in Config and re-run.')
else:
    print(f'Spike coincidence quality ({len(quick_spikes)} spikes):')
    print(f'{"#":>3}  {"Time ms":>8}  {"RMS":>10}  {"Delay ms":>10}  {"Quality":>8}  Note')
    print('-' * 65)
    for idx, s in enumerate(quick_spikes):
        flag = '  <-- NOISE?' if s['quality'] < QUALITY_WARN else ''
        print(f"{idx+1:>3}  {s['t_ms']:>8.1f}  {s['rms']:>10.5f}  "
              f"{s['delay_ms']:>10.3f}  {s['quality']:>8.3f}{flag}")

    # Plot quality scores
    times   = [s['t_ms']   for s in quick_spikes]
    quals   = [s['quality'] for s in quick_spikes]
    delays  = [s['delay_ms'] for s in quick_spikes]

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=('Coincidence Quality (1.0 = simultaneous)',
                                        'Inter-Channel Delay (ms)'))
    fig.add_trace(go.Scatter(x=times, y=quals, mode='markers+lines',
                  marker=dict(size=8, color='#66bb6a'), name='Quality'), row=1, col=1)
    fig.add_hline(y=QUALITY_WARN, line=dict(color='#ef5350', dash='dash'),
                  annotation_text='Noise flag threshold', row=1, col=1)
    fig.add_trace(go.Scatter(x=times, y=delays, mode='markers+lines',
                  marker=dict(size=8, color='#ffa726'), name='Delay ms'), row=2, col=1)
    fig.add_hline(y=0, line=dict(color='#555', width=1), row=2, col=1)
    fig.update_layout(title='Per-Spike Coincidence Quality', height=450, **DARK)
    fig.update_xaxes(title_text='Time (ms)', row=2, col=1)
    fig.show()

## Spike Detection + Lookback Pairing
Mirrors `detector.py` / `CycleTracker` exactly. FFT gate, debounce, T_start lookback.

In [ ]:
def compute_hf_energy(chunk, bin_low, bin_high):
    N = len(chunk)
    spectrum = np.abs(np.fft.rfft(chunk)) / N
    hi = min(bin_high, len(spectrum) - 1)
    return float(np.sum(spectrum[bin_low:hi+1]))

spikes = []
last_spike_chunk = -999
debounce_chunks  = DEBOUNCE_MS / CHUNK_MS

for i in range(n_chunks):
    rms = rms_vals[i]
    if rms < threshold:
        continue
    chunk_data = combined[i*chunk_samples:(i+1)*chunk_samples]
    hf = compute_hf_energy(chunk_data, HF_BIN_LOW, HF_BIN_HIGH)
    if hf < HF_FLOOR:
        spikes.append({'i': i, 'rms': rms, 'hf': hf, 'gated': True})
        continue
    if i - last_spike_chunk < debounce_chunks:
        continue
    last_spike_chunk = i
    spikes.append({'i': i, 'rms': rms, 'hf': hf, 'gated': False})

live_spikes = [s for s in spikes if not s['gated']]
gated       = [s for s in spikes if s['gated']]
print(f'Spikes above threshold: {len(live_spikes)}  |  Gated by FFT: {len(gated)}')

# ── Lookback pairing ──────────────────────────────────────────────────────────
min_chunks = MIN_LOOKBACK_MS / CHUNK_MS
max_chunks = MAX_LOOKBACK_MS / CHUNK_MS
cycles = []

for idx, tend_spike in enumerate(live_spikes):
    tend_i = tend_spike['i']
    best = None
    for prev in live_spikes[:idx]:
        gap = tend_i - prev['i']
        if gap < min_chunks or gap > max_chunks:
            continue
        if prev['rms'] < bkwy_thr:
            continue
        if best is None or prev['rms'] > best['rms']:
            best = prev
    if best is not None:
        delta_ms = (tend_i - best['i']) * CHUNK_MS
        speed    = (STROKE_IN / delta_ms) * 1000
        cycles.append({
            'tstart_i':   best['i'],      'tend_i':      tend_i,
            'delta_ms':   delta_ms,       'speed':       speed,
            'tstart_rms': best['rms'],    'tend_rms':    tend_spike['rms']
        })

print(f'\nCycles detected: {len(cycles)}')
for c in cycles:
    print(f"  T_start={c['tstart_i']*CHUNK_MS:.0f}ms  "
          f"T_end={c['tend_i']*CHUNK_MS:.0f}ms  "
          f"Δ={c['delta_ms']:.1f}ms  speed={c['speed']:.3f} in/s")

if cycles:
    deltas = [c['delta_ms'] for c in cycles]
    speeds = [c['speed']    for c in cycles]
    print(f'\nSession summary:')
    print(f'  Mean stroke:  {np.mean(deltas):.2f} ms')
    print(f'  SD:           {np.std(deltas):.2f} ms')
    print(f'  CV:           {np.std(deltas)/np.mean(deltas)*100:.2f} %')
    print(f'  Mean speed:   {np.mean(speeds):.3f} in/s')

## Detection Map — full picture
T_start (orange), T_end (green), gated events (grey dashed), shaded cycle windows with delta and speed.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_chunks, y=rms_vals, mode='lines',
              line=dict(color='#4fc3f7', width=1), name='RMS', opacity=0.8))
fig.add_hline(y=threshold, line=dict(color='#ef5350', width=1, dash='dash'),
              annotation_text='Impact threshold')
fig.add_hline(y=bkwy_thr,  line=dict(color='#ffa726', width=1, dash='dot'),
              annotation_text='Breakaway threshold')
fig.add_hline(y=baseline,  line=dict(color='#444444', width=1),
              annotation_text='Baseline')

for s in gated:
    fig.add_vline(x=s['i']*CHUNK_MS,
                  line=dict(color='#666666', width=0.8, dash='dot'))

for c in cycles:
    ts_ms = c['tstart_i'] * CHUNK_MS
    te_ms = c['tend_i']   * CHUNK_MS
    fig.add_vline(x=ts_ms, line=dict(color='#ffa726', width=2),
                  annotation_text='T_start')
    fig.add_vline(x=te_ms, line=dict(color='#66bb6a', width=2),
                  annotation_text='T_end')
    fig.add_vrect(x0=ts_ms, x1=te_ms, fillcolor='#66bb6a', opacity=0.08,
                  line_width=0,
                  annotation_text=f"Δ{c['delta_ms']:.0f}ms  {c['speed']:.1f}in/s",
                  annotation_position='top left',
                  annotation_font_color='#ffffff')

fig.update_layout(title='Detection Map',
                  xaxis_title='Time (ms)', yaxis_title='RMS', height=450, **DARK)
fig.show()

print(f'\nSummary: {len(cycles)} cycle(s) detected')
for c in cycles:
    print(f"  Δ={c['delta_ms']:.1f}ms  speed={c['speed']:.3f} in/s  "
          f"T_start RMS={c['tstart_rms']:.5f}  T_end RMS={c['tend_rms']:.5f}")

## Spike Browser — waveform + FFT per spike
Shows the raw waveform window and FFT for every detected spike.
Use to visually confirm T_start and T_end are real mechanical events.

In [ ]:
if not live_spikes:
    print('No spikes detected — lower IMPACT_MULT or check mounting.')
else:
    print(f'{len(live_spikes)} spike(s) — waveform + FFT for each\n')
    hf_low_khz  = HF_BIN_LOW  * sr / chunk_samples / 1000
    hf_high_khz = HF_BIN_HIGH * sr / chunk_samples / 1000

    for si, spike in enumerate(live_spikes):
        i         = spike['i']
        t_ms      = i * CHUNK_MS
        win_start = max(0, i - 2) * chunk_samples
        win_end   = min(len(combined), (i + 4) * chunk_samples)
        window    = combined[win_start:win_end]
        t_window  = np.linspace((i-2)*CHUNK_MS, (i+4)*CHUNK_MS, len(window))

        chunk_data = combined[i*chunk_samples:(i+1)*chunk_samples]
        freqs = np.fft.rfftfreq(len(chunk_data), d=1/sr) / 1000
        mags  = np.abs(np.fft.rfft(chunk_data)) / len(chunk_data)

        fig = make_subplots(rows=1, cols=2,
            subplot_titles=(
                f'Spike {si+1} waveform @ {t_ms:.0f}ms',
                f'Spike {si+1} FFT  (RMS={spike["rms"]:.5f}  HF={spike["hf"]:.5f})'))

        fig.add_trace(go.Scatter(x=t_window, y=window, mode='lines',
                      line=dict(color='#4fc3f7', width=0.8), name='signal'), row=1, col=1)
        fig.add_vline(x=t_ms,
                      line=dict(color='#ef5350', width=1.5, dash='dash'), row=1, col=1)
        fig.add_trace(go.Scatter(x=freqs, y=mags, mode='lines',
                      line=dict(color='#ffa726', width=0.8), name='magnitude'), row=1, col=2)
        fig.add_vrect(x0=hf_low_khz, x1=hf_high_khz,
                      fillcolor='#ffa726', opacity=0.1, line_width=0,
                      annotation_text='HF gate', row=1, col=2)

        fig.update_layout(height=350, showlegend=False, **DARK)
        fig.update_xaxes(title_text='Time (ms)',      row=1, col=1, gridcolor='#2a2a2a')
        fig.update_xaxes(title_text='Frequency (kHz)', row=1, col=2, gridcolor='#2a2a2a')
        fig.update_yaxes(gridcolor='#2a2a2a')
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

## Power Spectral Density — T_start vs T_end
Welch PSD shows the frequency signature of each event type.
Used to tune HF_FLOOR and as the foundation for future seal-wear frequency analysis
(per Shanbhag et al. — mean frequency and median frequency shift with wear).

In [ ]:
if not cycles:
    print('No cycles detected — run Spike Detection cell first.')
else:
    fig = go.Figure()
    colors_start = ['#ffa726', '#fb8c00', '#e65100']
    colors_end   = ['#66bb6a', '#43a047', '#2e7d32']

    for ci, c in enumerate(cycles):
        # T_start window
        ts_center = int(c['tstart_i'] * chunk_samples + chunk_samples // 2)
        ts_start  = max(0, ts_center - chunk_samples * 2)
        ts_end    = min(len(combined), ts_center + chunk_samples * 2)
        seg_start = combined[ts_start:ts_end]

        # T_end window
        te_center = int(c['tend_i'] * chunk_samples + chunk_samples // 2)
        te_start  = max(0, te_center - chunk_samples * 2)
        te_end    = min(len(combined), te_center + chunk_samples * 2)
        seg_end   = combined[te_start:te_end]

        f_s, psd_s = signal.welch(seg_start, fs=sr, nperseg=min(256, len(seg_start)))
        f_e, psd_e = signal.welch(seg_end,   fs=sr, nperseg=min(256, len(seg_end)))

        col_s = colors_start[ci % len(colors_start)]
        col_e = colors_end[ci % len(colors_end)]

        fig.add_trace(go.Scatter(x=f_s/1000, y=10*np.log10(psd_s + 1e-12),
                      mode='lines', line=dict(color=col_s, width=1.5),
                      name=f'Cycle {ci+1} T_start'))
        fig.add_trace(go.Scatter(x=f_e/1000, y=10*np.log10(psd_e + 1e-12),
                      mode='lines', line=dict(color=col_e, width=1.5, dash='dash'),
                      name=f'Cycle {ci+1} T_end'))

        # Mean and median frequency
        mean_f_s = np.sum(f_s * psd_s) / (np.sum(psd_s) + 1e-12)
        mean_f_e = np.sum(f_e * psd_e) / (np.sum(psd_e) + 1e-12)
        print(f'Cycle {ci+1}  T_start mean freq: {mean_f_s:.0f} Hz  '
              f'T_end mean freq: {mean_f_e:.0f} Hz')

    hf_low_hz  = HF_BIN_LOW  * sr / chunk_samples
    hf_high_hz = HF_BIN_HIGH * sr / chunk_samples
    fig.add_vrect(x0=hf_low_hz/1000, x1=hf_high_hz/1000,
                  fillcolor='#ffffff', opacity=0.04, line_width=0,
                  annotation_text='HF gate')

    fig.update_layout(title='Power Spectral Density — T_start vs T_end (dB)',
                      xaxis_title='Frequency (kHz)', yaxis_title='PSD (dB)',
                      height=450, **DARK)
    fig.show()

## Suggested App Settings
Auto-calculates optimal multipliers and HF Floor from this recording.
Copy these values back into the Config cell and into the PWA Settings tab.

In [ ]:
if not live_spikes:
    print('No spikes detected — lower IMPACT_MULT or check mounting.')
else:
    rms_values   = [s['rms'] for s in live_spikes]
    hf_values    = [s['hf']  for s in live_spikes]
    rms_min      = min(rms_values)
    rms_min_mult = rms_min / baseline
    hf_min       = min(hf_values)
    MARGIN       = 0.6

    sug_impact    = max(1.5, round(rms_min_mult * MARGIN, 1))
    sug_breakaway = max(1.0, round(sug_impact * 0.5, 1))
    sug_hf        = round(hf_min * 0.8, 4)

    print('── Suggested settings (copy to Config and app Settings tab) ──')
    print(f'  IMPACT_MULT:     {sug_impact}')
    print(f'  BREAKAWAY_MULT:  {sug_breakaway}')
    print(f'  HF_FLOOR:        {sug_hf}')
    print(f'  (Based on {MARGIN*100:.0f}% of weakest event, baseline={baseline:.6f})')

    if cycles:
        deltas = [c['delta_ms'] for c in cycles]
        print(f'\n── Lookback window recommendation ──')
        print(f'  Measured delta range: {min(deltas):.1f} – {max(deltas):.1f} ms')
        print(f'  Suggested MIN_LOOKBACK_MS: {max(5, int(min(deltas)*0.5))}')
        print(f'  Suggested MAX_LOOKBACK_MS: {int(max(deltas)*1.5)}')

## Log Session — append results to CSV
Stores cylinder ID, PSI, date, stroke times, mean, SD, speed.
This is the data foundation for trend analysis. Run once per session after reviewing cycles.

In [ ]:
LOG_PATH = Path(SESSION_LOG_CSV)
LOG_PATH.parent.mkdir(exist_ok=True)

FIELDNAMES = [
    'timestamp', 'cylinder_id', 'psi', 'n_cycles',
    'mean_ms', 'sd_ms', 'cv_pct', 'mean_speed_ips',
    'min_ms', 'max_ms', 'deltas_ms', 'notes'
]

if not cycles:
    print('No cycles detected — nothing to log. Run Spike Detection first.')
else:
    deltas = [c['delta_ms'] for c in cycles]
    speeds = [c['speed']    for c in cycles]

    row = {
        'timestamp':      datetime.datetime.now().isoformat(timespec='seconds'),
        'cylinder_id':    CYLINDER_ID,
        'psi':            PSI,
        'n_cycles':       len(cycles),
        'mean_ms':        round(np.mean(deltas), 3),
        'sd_ms':          round(np.std(deltas),  3),
        'cv_pct':         round(np.std(deltas) / np.mean(deltas) * 100, 3),
        'mean_speed_ips': round(np.mean(speeds), 4),
        'min_ms':         round(min(deltas), 3),
        'max_ms':         round(max(deltas), 3),
        'deltas_ms':      '|'.join(f'{d:.2f}' for d in deltas),
        'notes':          NOTES,
    }

    write_header = not LOG_PATH.exists()
    with open(LOG_PATH, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

    print(f'Logged to: {LOG_PATH}')
    print(f'  Cylinder {CYLINDER_ID} | PSI {PSI} | {len(cycles)} cycles')
    print(f'  Mean: {row["mean_ms"]} ms | SD: {row["sd_ms"]} ms | CV: {row["cv_pct"]}%')
    print(f'  Speed: {row["mean_speed_ips"]} in/s')

    # Show full log so far
    df = pd.read_csv(LOG_PATH)
    print(f'\nFull log ({len(df)} sessions):')
    display(df[['timestamp','cylinder_id','psi','n_cycles','mean_ms','sd_ms','mean_speed_ips']].tail(10))

    date_str = datetime.datetime.now().strftime('%Y-%m-%d')
    print(f'\n── Commit to save permanently ──')
    print(f'  git add session_logs/cylinder_sessions.csv')
    print(f'  git commit -m "data: cylinder session {date_str} — cyl {CYLINDER_ID}, {len(cycles)} cycles, {row[\"mean_ms\"]}ms mean"')

## Cross-Session Trend Plot
Plots stroke time over all logged sessions. Fits a linear trend line.
A rising slope = increasing stroke time = potential seal wear.
Requires at least 2 logged sessions.

In [ ]:
LOG_PATH = Path(SESSION_LOG_CSV)

if not LOG_PATH.exists():
    print(f'No session log found at {LOG_PATH}. Run Log Session cell after measuring.')
else:
    df = pd.read_csv(LOG_PATH)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)

    if len(df) < 2:
        print('Need at least 2 sessions to plot a trend. Keep measuring!')
    else:
        fig = go.Figure()
        colors_cyl = {1: '#4fc3f7', 2: '#66bb6a', 3: '#ffa726', 4: '#ef5350'}

        for cyl_id in sorted(df['cylinder_id'].unique()):
            cdf = df[df['cylinder_id'] == cyl_id].copy()
            cdf['session_n'] = range(len(cdf))
            col = colors_cyl.get(cyl_id, '#ffffff')

            fig.add_trace(go.Scatter(
                x=cdf['timestamp'], y=cdf['mean_ms'],
                mode='markers+lines',
                error_y=dict(type='data', array=cdf['sd_ms'].tolist(), visible=True),
                line=dict(color=col, width=1.5),
                marker=dict(size=8, color=col),
                name=f'Cylinder {cyl_id}'
            ))

            # Trend line
            if len(cdf) >= 3:
                x_num = cdf['session_n'].values
                y     = cdf['mean_ms'].values
                slope, intercept, r, p, _ = stats.linregress(x_num, y)
                trend_y = slope * x_num + intercept
                fig.add_trace(go.Scatter(
                    x=cdf['timestamp'], y=trend_y,
                    mode='lines', line=dict(color=col, width=1, dash='dash'),
                    name=f'Cyl {cyl_id} trend (slope={slope:+.3f} ms/session, R²={r**2:.3f})',
                    showlegend=True
                ))
                trend_dir = 'INCREASING ↑' if slope > 0 else 'decreasing ↓'
                sig = '(significant p<0.05)' if p < 0.05 else f'(p={p:.2f}, not yet significant)'
                print(f'Cylinder {cyl_id}: slope={slope:+.4f} ms/session — {trend_dir} {sig}')

        fig.update_layout(title='Stroke Time Trend — All Sessions',
                          xaxis_title='Date', yaxis_title='Mean Stroke Time (ms)',
                          height=500, **DARK)
        fig.show()

## CUSUM Alert — sustained shift detection
Detects a sustained upward shift in stroke time — the standard industrial algorithm
for predictive maintenance alerting. More sensitive than a simple threshold on mean.
A CUSUM flag means: 'this cylinder has been consistently slower for several sessions'.
Tune CUSUM_K and CUSUM_H in Config to adjust sensitivity.

In [ ]:
def cusum(data, k=0.5, h=5.0):
    """One-sided upper CUSUM. Returns (s_pos array, flag array).
    k = slack (half the shift size in std units you want to detect)
    h = decision threshold (raise to reduce false alarms)
    """
    n      = len(data)
    mu     = np.mean(data[:max(2, n//4)])   # use first quarter as reference
    sigma  = np.std(data[:max(2, n//4)]) or 1.0
    s_pos  = np.zeros(n)
    flags  = np.zeros(n, dtype=bool)
    for i in range(1, n):
        s_pos[i] = max(0, s_pos[i-1] + (data[i] - mu) / sigma - k)
        flags[i] = s_pos[i] > h
    return s_pos, flags

LOG_PATH = Path(SESSION_LOG_CSV)

if not LOG_PATH.exists():
    print(f'No session log found. Run Log Session cell first.')
else:
    df = pd.read_csv(LOG_PATH)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)

    if len(df) < 4:
        print(f'CUSUM needs at least 4 sessions — you have {len(df)}. Keep measuring!')
    else:
        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            subplot_titles=('Stroke Time (ms)', 'CUSUM Statistic'))
        colors_cyl = {1: '#4fc3f7', 2: '#66bb6a', 3: '#ffa726', 4: '#ef5350'}

        for cyl_id in sorted(df['cylinder_id'].unique()):
            cdf = df[df['cylinder_id'] == cyl_id].sort_values('timestamp')
            if len(cdf) < 4:
                continue
            col    = colors_cyl.get(cyl_id, '#ffffff')
            times  = cdf['timestamp'].tolist()
            y      = cdf['mean_ms'].values
            s_pos, flags = cusum(y, k=CUSUM_K, h=CUSUM_H)

            fig.add_trace(go.Scatter(x=times, y=y, mode='markers+lines',
                          line=dict(color=col, width=1.5),
                          marker=dict(size=8, color=col),
                          name=f'Cyl {cyl_id}'), row=1, col=1)

            fig.add_trace(go.Scatter(x=times, y=s_pos, mode='lines',
                          line=dict(color=col, width=1.5),
                          name=f'Cyl {cyl_id} CUSUM'), row=2, col=1)

            # Flag first breach
            breach_idx = np.where(flags)[0]
            if len(breach_idx) > 0:
                breach_t = times[breach_idx[0]]
                fig.add_vline(x=str(breach_t),
                              line=dict(color='#ef5350', width=2, dash='dash'),
                              annotation_text=f'ALERT Cyl {cyl_id}',
                              annotation_font_color='#ef5350')
                print(f'CUSUM ALERT — Cylinder {cyl_id}: sustained upward shift '
                      f'detected at session {breach_idx[0]+1} ({breach_t})')
            else:
                print(f'Cylinder {cyl_id}: no CUSUM alert — within normal range.')

        fig.add_hline(y=CUSUM_H, line=dict(color='#ef5350', width=1, dash='dash'),
                      annotation_text=f'Alert threshold h={CUSUM_H}', row=2, col=1)
        fig.update_layout(title=f'CUSUM — k={CUSUM_K}, h={CUSUM_H}',
                          height=550, **DARK)
        fig.show()

## Inter-Cylinder Ratio
Tracks each cylinder's stroke time relative to the mean of all 4 per session.
Normalises out pressure fluctuations that affect all cylinders equally.
A diverging ratio is the earliest possible warning of single-cylinder wear —
it appears before absolute stroke time moves measurably.
Requires sessions from multiple cylinders in the log.

In [ ]:
LOG_PATH = Path(SESSION_LOG_CSV)

if not LOG_PATH.exists():
    print(f'No session log found. Run Log Session cell first.')
else:
    df = pd.read_csv(LOG_PATH)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').reset_index(drop=True)

    n_cyls = df['cylinder_id'].nunique()
    if n_cyls < 2:
        print(f'Need sessions from at least 2 cylinders — you have {n_cyls}.')
        print('Measure more cylinders and run Log Session for each.')
    else:
        # Group sessions by date (day-level) to align cylinder measurements
        df['date'] = df['timestamp'].dt.date
        pivot = df.groupby(['date', 'cylinder_id'])['mean_ms'].mean().unstack('cylinder_id')
        pivot.columns = [f'Cyl {c}' for c in pivot.columns]

        # Ratio = each cylinder / mean of all cylinders that day
        daily_mean = pivot.mean(axis=1)
        ratio = pivot.div(daily_mean, axis=0)

        print('Inter-cylinder ratio (1.0 = at fleet mean, >1.0 = slower than peers):')
        display(ratio.round(4))

        # Plot
        colors_cyl = {f'Cyl {i}': c for i, c in
                      zip([1,2,3,4], ['#4fc3f7','#66bb6a','#ffa726','#ef5350'])}
        fig = go.Figure()

        for col in ratio.columns:
            color = colors_cyl.get(col, '#ffffff')
            fig.add_trace(go.Scatter(
                x=ratio.index.astype(str), y=ratio[col],
                mode='markers+lines',
                line=dict(color=color, width=1.5),
                marker=dict(size=8, color=color),
                name=col
            ))

        fig.add_hline(y=1.0, line=dict(color='#555', width=1),
                      annotation_text='Fleet mean')
        fig.add_hrect(y0=0.95, y1=1.05, fillcolor='#ffffff',
                      opacity=0.03, line_width=0,
                      annotation_text='±5% band')
        fig.update_layout(
            title='Inter-Cylinder Ratio (stroke time vs fleet mean)',
            xaxis_title='Date', yaxis_title='Ratio',
            height=450, **DARK
        )
        fig.show()

        # Flag any cylinder consistently above 1.05
        for col in ratio.columns:
            pct_high = (ratio[col] > 1.05).mean() * 100
            if pct_high > 50:
                print(f'WARNING: {col} is above fleet mean >5% in {pct_high:.0f}% of sessions — investigate.')
            else:
                print(f'{col}: OK ({pct_high:.0f}% of sessions above +5% band)')